In [ ]:
import pandas as pd
from pathlib import Path
import re
import numpy as np
import pandas as pd

In [ ]:

BASE_DIR = Path.cwd().parent
STAGE2_DIR = BASE_DIR / "data" / "processed" / "stage2"

stage2_train_df = pd.read_csv(STAGE2_DIR / "stage2_train_df.csv")
stage2_val_df   = pd.read_csv(STAGE2_DIR / "stage2_val_df.csv")

assert set(stage2_train_df.columns) == set(stage2_val_df.columns), "train/val columns differ"
stage2_val_df = stage2_val_df[stage2_train_df.columns]

print("TRAIN:", stage2_train_df.shape, "| VAL:", stage2_val_df.shape)
display(stage2_train_df.head())

TRAIN: (5165, 60) | VAL: (1252, 60)


,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,nokia_model,lenovo_model,lg_model,model_key,match_rule,confidence,family,storage_known,ram_known,condition_known
0,ايفون 13 iPhone,19500.0,مساكن المهندسين، مدينة نصر,/ad/%D8%A7%D9%8A%D9%81%D9%88%D9%86-13-iphone-I...,208854767,legacy,apple,NaN,NaN,NaN,...,NaN,NaN,NaN,apple iphone 13,apple:numeric_pattern,1.0,iphone,False,False,False
1,Honor Magic V5,80000.0,اوبن اير مول، مدينتي,/ad/honor-magic-v5-ID209010621.html,209010621,legacy,honor,NaN,NaN,NaN,...,NaN,NaN,NaN,honor magic v5,honor:rule_05,NaN,magic,False,False,False
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,57999.0,سموحة، الإسكندرية,/ad/oppo-find-x9-pro-%D8%B6%D9%85%D8%A7%D9%86-...,209003975,legacy,oppo,NaN,NaN,NaN,...,NaN,NaN,NaN,oppo find x9 pro,oppo:rule_03,NaN,find,False,False,False
3,Honor Robot Phone 512G 12G ram,94000.0,مدينة نصر، القاهرة,/ad/honor-robot-phone-512g-12g-ram-ID208841945...,208841945,legacy,honor,512.0,12.0,L1:unit,...,NaN,NaN,NaN,honor robot,honor:rule_11,NaN,robot,True,True,False
4,شاومي 17 الترا xiaomi 17 ultra,70000.0,التجمع الخامس، القاهرة الجديدة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-17-%D8%A7%D...,208281312,legacy,xiaomi,NaN,NaN,NaN,...,NaN,NaN,NaN,xiaomi 17 ultra,xiaomi:rule_17,NaN,xiaomi,False,False,False


In [ ]:
# ───────────── 1) Text normalisation ─────────────
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u0640]")  # tashkeel + tatweel
INVISIBLES = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]")

def norm_text(s) -> str:
    if pd.isna(s):
        return ""
    s = str(s).translate(AR_DIGITS)
    s = INVISIBLES.sub(" ", s)
    s = AR_DIACRITICS.sub("", s)
    s = s.lower()
    s = (s.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
           .replace("ى", "ي").replace("ة", "ه").replace("ؤ", "و").replace("ئ", "ي"))
    s = s.replace("٪", "%").replace("＋", "+").replace("×", "x")
    s = re.sub(r"(\d)[,،](\d{3})(?!\d)", r"\1\2", s)   # 1,024 -> 1024
    s = re.sub(r"[|_•·●▪■★☆✔✅🔥💥⭐]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
# ───────────── 2) Storage / RAM ─────────────
STORAGE_SIZES = {8, 16, 32, 64, 128, 256, 512, 1024}
RAM_SIZES     = {1, 2, 3, 4, 6, 8, 10, 12, 16, 18, 20, 24}

# unit words (English + Arabic/Egyptian spellings, after normalisation)
GB_UNIT = r"(?:gb|g\.?b|gig(?:a)?(?:byte)?s?|giga|جيجا(?:بايت)?|جيجه|جيجا|جي ?بي|جيبي|جب|غيغا(?:بايت)?|قيقا)"
TB_UNIT = r"(?:tb|t\.?b|tera(?:byte)?s?|تيرا(?:بايت)?|ترا بايت)"
NOT_AFTER = r"(?!\s*(?:mp|ميجا|مب|mah|ملي|هرتز|hz|w\b|watt|وات|inch|بوصه|\"|ghz|nm|fps|%))"
NUM_L = r"(?<![\d.,/+])"
RAM_KW = r"(?:ram|رامات|رام|ذاكره عشوائيه|عشوائيه|memory|ميمورى|ميموري)"
ROM_KW = r"(?:rom|storage|تخزين|مساحه|مساحة|سعه|ذاكره داخليه|ذاكره|internal|ستورج|ستوريج|هارد|ssd)"

RE_TB = re.compile(NUM_L + r"(\d(?:\.\d)?)\s*" + TB_UNIT + r"(?![a-z])")
# 8/256 , 256/8 , 8+256 , 8gb/256gb , 8 g / 256 g , 12 - 512 , 8 x 256
RE_PAIR = re.compile(   # lookahead => overlapping matches ("s26+ 12/256" finds 12/256, not 26+12)
    NUM_L + r"(?=(\d{1,4})\s*(?:" + GB_UNIT + r")?\s*(?:/|\\|\+|\-|x|،|,|\|)\s*(\d{1,4})\s*(?:" + GB_UNIT + r")?(?![\d.]|\s*(?:mp|mah|hz|ميجا)))")
# 256gb / 256 جيجا / 256g (g only validated against storage sizes so 5g / 4g never match)
RE_STORAGE_UNIT = re.compile(NUM_L + r"(\d{1,4})\s*" + GB_UNIT + r"(?![a-z])" + NOT_AFTER)
RE_STORAGE_G = re.compile(NUM_L + r"(\d{2,4})\s*g(?![a-z\d])")                     # "256g"
RE_STORAGE_KW_AFTER  = re.compile(ROM_KW + r"\s*[:=\-]?\s*(\d{1,4})(?!\d)" + NOT_AFTER)
RE_STORAGE_KW_BEFORE = re.compile(NUM_L + r"(\d{1,4})\s*(?:" + GB_UNIT + r")?\s*" + ROM_KW)
RE_STORAGE_BARE = re.compile(NUM_L + r"(64|128|256|512)(?![\d.,]|\s*(?:mp|mah|hz|ميجا|w\b|مب|%))(?![a-z]{3,})")
RE_STORAGE_BARE_1TB = re.compile(NUM_L + r"(1024)(?!\d)")

RE_RAM_KW_AFTER  = re.compile(RAM_KW + r"\s*[:=\-]?\s*(\d{1,2})(?!\d)\s*(?:" + GB_UNIT + r"|g)?" + NOT_AFTER)
RE_RAM_KW_BEFORE = re.compile(NUM_L + r"(\d{1,2})\s*(?:" + GB_UNIT + r"|g)?\s*(?:\+\s*\d{1,2}\s*(?:" + GB_UNIT + r"|g)?\s*)?" + RAM_KW)
RE_PAIR_SPACE    = re.compile(NUM_L + r"(\d{1,2})\s*(?:gb|g|جيجا)\.?\s+(\d{2,4})\s*(?:gb|g|جيجا|ج)?(?![\d.]|\s*(?:mp|mah|hz|ميجا))")  # "12g 256g"
RE_PAIR_PLUS_ROM = re.compile(r"(?:رام|رامات|ram)\s*(\d{1,2})\s*\+\s*(\d{1,2}).{0,25}?(?:128|256|512|64|32)")          # "رام 8+4 ..."
RE_RAM_EXT       = re.compile(NUM_L + r"(\d{1,2})\s*(?:" + GB_UNIT + r"|g)?\s*\+\s*(\d{1,2})\s*(?:" + GB_UNIT + r"|g)?\s*(?:ram|رام)")  # 8+8 ram (virtual)

def _to_gb(val: float) -> int:
    return int(val)

def extract_storage_ram(raw_title):
    """returns dict(storage_gb, ram_gb, storage_src, ram_src, storage_ambiguous, ram_ambiguous)"""
    t = norm_text(raw_title)
    out = dict(storage_gb_v2=np.nan, ram_gb_v2=np.nan, storage_src_v2=None, ram_src_v2=None,
               storage_ambiguous=False, ram_ambiguous=False)
    if not t:
        return out
    storage, ram = [], []            # (value, src)

    # --- TB ---
    for m in RE_TB.finditer(t):
        v = float(m.group(1)) * 1024
        if v in (1024, 2048, 4096):
            storage.append((int(v), "tb"))

    # --- pairs  RAM/ROM ---
    for m in RE_PAIR.finditer(t):
        a, b = int(m.group(1)), int(m.group(2))
        if a in RAM_SIZES and b in STORAGE_SIZES and b > a:
            ram.append((a, "pair")); storage.append((b, "pair"))
        elif b in RAM_SIZES and a in STORAGE_SIZES and a > b:
            ram.append((b, "pair")); storage.append((a, "pair"))
        elif a in STORAGE_SIZES and b in STORAGE_SIZES and a != b:
            # "128/256" => two variants listed (ambiguous) -> keep both as candidates
            storage.append((a, "pair_multi")); storage.append((b, "pair_multi"))

    for m in RE_PAIR_SPACE.finditer(t):
        a, b = int(m.group(1)), int(m.group(2))
        if a in RAM_SIZES and b in STORAGE_SIZES and b > a:
            ram.append((a, "pair")); storage.append((b, "pair"))
    for m in RE_PAIR_PLUS_ROM.finditer(t):          # "رامات 4+2" → physical RAM = first number
        a = int(m.group(1))
        if a in RAM_SIZES:
            ram.append((a, "ext"))

    # --- explicit RAM keywords ---
    for rx, src in ((RE_RAM_KW_AFTER, "kw"), (RE_RAM_KW_BEFORE, "kw")):
        for m in rx.finditer(t):
            v = int(m.group(1))
            if v in RAM_SIZES:
                ram.append((v, src))
    for m in RE_RAM_EXT.finditer(t):          # 8+8 ram → physical RAM is the first
        v = int(m.group(1))
        if v in RAM_SIZES:
            ram.append((v, "ext"))

    # --- storage with units / keywords ---
    for m in RE_STORAGE_UNIT.finditer(t):
        v = int(m.group(1))
        if v in STORAGE_SIZES:
            ctx = t[m.end():m.end() + 8]
            if re.match(r"\s*" + RAM_KW, ctx):      # "8gb ram" is RAM, not storage
                continue
            storage.append((v, "unit"))
    for m in RE_STORAGE_G.finditer(t):
        v = int(m.group(1))
        if v in STORAGE_SIZES and v >= 32:
            storage.append((v, "g"))
    for rx in (RE_STORAGE_KW_AFTER, RE_STORAGE_KW_BEFORE):
        for m in rx.finditer(t):
            v = int(m.group(1))
            if v in STORAGE_SIZES and v >= 32:
                storage.append((v, "kw"))

    # --- bare numbers (last resort) ---
    if not storage:
        for m in RE_STORAGE_BARE.finditer(t):
            storage.append((int(m.group(1)), "bare"))
        for m in RE_STORAGE_BARE_1TB.finditer(t):
            storage.append((1024, "bare"))

    # --- RAM-vs-storage clean-up ---
    ram_vals = {v for v, _ in ram}
    if ram_vals:   # a number already claimed as RAM (e.g. "8gb ram") must not also be storage
        storage = [(v, s_) for v, s_ in storage if not (v in ram_vals and v < 32)]
    # when both a big (>=32) and a small (<32) size appear with units and no RAM keyword: "8Gb 256Gb" -> RAM/ROM
    small = [(v, s_) for v, s_ in storage if v < 32]
    big   = [(v, s_) for v, s_ in storage if v >= 32]
    if small and big and not ram:
        sv = sorted({v for v, _ in small})
        if len(sv) == 1 and sv[0] in RAM_SIZES:
            ram.append((sv[0], "unit_pair"))
        storage = big
    elif small and not big and not ram:
        pass       # "J3 2016 8GB" -> keep as storage (old phones have 8/16 GB storage)
    elif small and big:
        storage = big

    # --- resolve storage ---
    if storage:
        vals = sorted({v for v, _ in storage})
        if len(vals) == 1:
            out["storage_gb_v2"] = vals[0]
            out["storage_src_v2"] = next(s for v, s in storage if v == vals[0])
        else:
            out["storage_ambiguous"] = True      # several sizes in one title → don't guess
            out["storage_src_v2"] = "multi:" + "|".join(map(str, vals))
    # --- resolve ram ---
    if ram:
        vals = sorted({v for v, _ in ram})
        if len(vals) == 1:
            out["ram_gb_v2"] = vals[0]
            out["ram_src_v2"] = ram[0][1]
        else:
            out["ram_ambiguous"] = True
            out["ram_src_v2"] = "multi:" + "|".join(map(str, vals))
    return out

In [ ]:
# ───────────── 3) Condition ─────────────
# cleaned first: remove "new battery / new screen / new sim" etc. so they do not mean "new phone"
RE_NEW_PART = re.compile(
    r"(?:بطاري\w*|شاش\w*|بوردف?\w*|كاميرا|غطا\w*|علبه|كفر|جراب|سيم|شريحه|خط|شاحن\w*|سماع\w*|اكسسوار\w*|case|cover|battery|screen|charger|sim|box)\s*(?:\w+\s+){0,1}(?:جديد\w*|زيرو|new)"
    r"|(?:جديد\w*|زيرو|new)\s*(?:بطاري\w*|شاش\w*|بوردف?\w*|غطا\w*|كفر|جراب|شاحن\w*|battery|screen|charger|case|cover)"
    r"|(?:تغيير|تبديل|بدل|مغير|مبدل|changed|replaced)\s*(?:\w+\s+){0,1}(?:بطاري\w*|شاش\w*)")

RE_NEG_NEW = re.compile(r"(?:غير|مش|مو|ليس|not|non|لا|ماهو|مبقاش|بقي)\s*(?:جديد\w*|زيرو|new)")

RE_PARTS = re.compile(
    r"(?:للقطع|قطع غيار|معطل\w*|بايظ\w*|عطلان\w*|عطل\b|خربان\w*|(?<!\w)ميت(?!\s*(?:\d|اكس|x|برو|لايت|بود|ماكس|ماتش))|مش بيشتغل|لا يعمل|لا يشتغل|محتاج\s+(?:صيانه|شاشه|بوردف?|تصليح|اصلاح|تغيير|بطاريه|بورد)"
    r"|شاشه\s+(?:مكسور\w*|مكسوره|مشروخ\w*)|مكسور(?!\s*دهر)\w*|مشروخ\w*|هاوسينج|housing|for parts|broken|damaged|not working|dead|faulty|بدون شاشه|ايكلاود|icloud lock|مقفول ايكلاود|آي?كلاود|مغلق ايكلاود|فيه عطل|خلل)")

RE_LIKE_NEW = re.compile(
    r"(?:كسر\s*(?:ال)?زيرو|كسر\s*زيرو|كسرزيرو|كسر\s*(?:ال)?جديد|كالجديد|كاالجديد|شبه\s*(?:ال)?جديد|زي\s*(?:ال)?جديد|زى\s*(?:ال)?جديد|"
    r"(?:بحاله|حاله|حالته|حالتها)\s*(?:ال)?(?:الوكاله|كسر زيرو)|"
    r"like\s*new|likenew|open\s*box|openbox|mint(?:\s*condition)?|excellent condition|"
    r"(?:مستعمل|مستخدم)\s*(?:\w+\s+){0,1}(?:ساعتين|يومين|مره|مرتين)\s*(?:فقط|بس)?|"
    r"(?:نضيف|نظيف|فل|ممتاز\w*)\s*(?:جدا|اوي|قوي|خالص)?\s*(?:ك|زي)(?:ال)?(?:جديد|زيرو)|"
    r"(?:بحاله|حاله)\s*(?:الوكاله|المصنع|فبريكا|فابريكا)|"
    r"فبريكا|فابريكا|فرز\s*اول|فرز\s*اولي)")  # bare "كسر" last: "كسر" alone usually = "كسر زيرو" in this market

RE_NEW = re.compile(
    r"(?:\bnew\b|brand\s*new|sealed|(?:بحاله|حاله)\s*(?:ال)?(?:زيرو|جديد)|unopened|never\s*used|factory\s*sealed|"
    r"جديد\w*|زيرو|زيروو+|\bzero\b|0km|"
    r"متبرشم|متبرشمه|مبرشم|متقفل|مقفول|مقفوله|مغلف|مغلفه|متغلف|كارتونه?\s*مقفول\w*|بالبرشام|برشام|سيلد|"
    r"لم\s*(?:يستخدم|يستعمل|يفتح|يتم\s*(?:فتح|استخدام))|غير\s*(?:مستخدم|مستعمل|مفتوح)|متفتحش|مفتحش|مااستخدمش|مش\s*مستخدم|ما\s*اتفتحش|"
    r"لسه\s*(?:في|ب)\s*(?:ال)?(?:كرتونه?|علبه|كارتونه?|بلاستيك)|"
    r"(?:جيت|وصل|وارد)\s*(?:جديد|حديثا)|حديث(?:ه)?\s*الاستيراد|مدفوع\s*(?:ال)?(?:ضريبه|جمارك)\s*جديد)")

RE_USED = re.compile(
    r"(?:\bused\b|second\s*hand|secondhand|pre-?owned|refurb\w*|"
    r"مستعمل\w*|مستخدم\w*|استعمال\w*|استخدام\w*|متستعمل|ستعمل|"
    r"مجدد\w*|مجدده|مجددة|متجدد|"
    r"(?:فيه|فيها|عليه|عليها|بها|به)\s*(?:خدش|خدوش|شروخ|شرخ|علامات|عيب|سكراتش|scratch\w*|كسر)|"
    r"(?:اتصلح|اتغير|اتفتح)\w*|ساعات\s*استخدام|عمر\w*\s*(?:سنه|سنتين|شهر)|بطاريه\s*\d{2}\s*%|battery\s*health\s*\d{2}\s*%?|\d{2}\s*%\s*بطاريه|صحه\s*البطاريه)")

RE_NOISE = re.compile(
    r"(?:zero|زيرو|0|جديد)\s*(?:tax|ضريبه|ضرايب|جمارك|جمرك|scratch\w*|خدوش|خدش|خربوش)"      # "زيرو ضريبة" / "zero tax" is NOT a condition
    r"|(?:ضريبه|جمارك|tax|scratch\w*)\s*(?:زيرو|zero|0)"
    r"|(?:مش|بدون|مفيش|مفيهوش|مفهوش|no|without)\s*(?:عليه\s*)?(?:ضريبه|جمارك|tax)"
    r"|(?:battery|بطاريه|بطارية|شحن)\s*(?:health|صحه|هيلث)?\s*[:=\-]?\s*\d{2,3}\s*[%٪]?"
    r"|\d{2,3}\s*[%٪]\s*(?:battery|بطاريه|b\b)?"
    r"|\(\s*zero\s*\)")

# weak wording: "good / excellent condition" -> only means "used" when nothing says "new / never opened"
RE_USED_WEAK = re.compile(r"(?:ب)?حاله\s*(?:جيده|كويسه|ممتازه|نضيفه|عاديه|مقبوله|حلوه)")

def extract_condition(raw_title):
    """returns (condition, src)  – labels: new / like_new / used / parts / ambiguous"""
    t = norm_text(raw_title)
    if not t:
        return (None, None)
    t = RE_NEW_PART.sub(" ", t)
    t = RE_NOISE.sub(" ", t)
    parts   = bool(RE_PARTS.search(t))
    like    = bool(RE_LIKE_NEW.search(t))
    neg_new = bool(RE_NEG_NEW.search(t))
    t_wo_like = RE_LIKE_NEW.sub(" ", t)               # so "كسر زيرو" does not also count as new
    new     = bool(RE_NEW.search(RE_NEG_NEW.sub(" ", t_wo_like)))
    used    = bool(RE_USED.search(t_wo_like)) or neg_new          # strong evidence of use
    used_weak = bool(RE_USED_WEAK.search(t_wo_like))

    if parts:
        return ("parts", "rx:parts")
    if like and not used:
        return ("like_new", "rx:like_new")
    if like and used:
        return ("like_new", "rx:like_new+used")       # "مستعمل استعمال خفيف"
    if new and used:
        return ("ambiguous", "rx:new+used")
    if new:
        return ("new", "rx:new")                      # "بحالة جيدة لم يفتح" -> new (weak wording ignored)
    if used or used_weak:
        return ("used", "rx:used")
    return (None, None)

In [ ]:
# ───────────── 4) Apply to the dataframe (updates the EXISTING columns in place) ─────────────
def add_improved_features(df: pd.DataFrame, keep_debug: bool = False) -> pd.DataFrame:
    """Fills gaps in storage_gb / ram_gb / condition (+ their *_src) and recomputes the *_known flags.
    Existing stage-2 values are never overwritten. New values get src 'L2:<rule>'.
    No new columns are added unless keep_debug=True (storage_ambiguous / ram_ambiguous)."""
    df = df.copy()
    sr = pd.DataFrame([extract_storage_ram(t) for t in df["title"]], index=df.index)
    cond = df["title"].map(extract_condition)
    cond_v, cond_s = cond.map(lambda x: x[0]), cond.map(lambda x: x[1])

    m = df["storage_gb"].isna() & sr["storage_gb_v2"].notna()
    df.loc[m, "storage_gb"]  = sr.loc[m, "storage_gb_v2"]
    df.loc[m, "storage_src"] = "L2:" + sr.loc[m, "storage_src_v2"].astype(str)

    m = df["ram_gb"].isna() & sr["ram_gb_v2"].notna()
    df.loc[m, "ram_gb"]  = sr.loc[m, "ram_gb_v2"]
    df.loc[m, "ram_src"] = "L2:" + sr.loc[m, "ram_src_v2"].astype(str)

    m = df["condition"].isna() & cond_v.notna()
    df.loc[m, "condition"]     = cond_v[m]
    df.loc[m, "condition_src"] = "L2:" + cond_s[m].str.replace("rx:", "", regex=False)

    df["storage_known"]   = df["storage_gb"].notna()
    df["ram_known"]       = df["ram_gb"].notna()
    df["condition_known"] = df["condition"].notna() & (df["condition"] != "ambiguous")

    if keep_debug:
        df["storage_ambiguous"] = sr["storage_ambiguous"]
        df["ram_ambiguous"]     = sr["ram_ambiguous"]
    return df

## Apply to train and val

In [ ]:
train_old, val_old = stage2_train_df.copy(), stage2_val_df.copy()

stage2_train_df = add_improved_features(train_old)
stage2_val_df   = add_improved_features(val_old)

flags = ["storage_known", "ram_known", "condition_known"]
rows = []
for name, before, after in [("train", train_old, stage2_train_df), ("val", val_old, stage2_val_df)]:
    for f in flags:
        rows.append({"split": name, "flag": f, "before": int(before[f].sum()), "after": int(after[f].sum()),
                     "gain": int(after[f].sum() - before[f].sum()),
                     "coverage_before_%": round(before[f].mean() * 100, 2),
                     "coverage_after_%": round(after[f].mean() * 100, 2)})
display(pd.DataFrame(rows))


,split,flag,before,after,gain,coverage_before_%,coverage_after_%
0,train,storage_known,377,389,12,7.30,7.53
1,train,ram_known,119,141,22,2.30,2.73
2,train,condition_known,291,338,47,5.63,6.54
3,val,storage_known,93,97,4,7.43,7.75
4,val,ram_known,28,36,8,2.24,2.88
5,val,condition_known,63,75,12,5.03,5.99


### Spot-check the new fills on TRAIN (this is where rules get reviewed)

In [ ]:
def l2_rows(df):
    return df[df["storage_src"].str.startswith("L2", na=False)
              | df["ram_src"].str.startswith("L2", na=False)
              | df["condition_src"].str.startswith("L2", na=False)]

display(l2_rows(stage2_train_df)[["title","storage_gb","ram_gb","condition","storage_src","ram_src","condition_src"]].head(40))

,title,storage_gb,ram_gb,condition,storage_src,ram_src,condition_src
10,Oppo Find X5 5G dual sim 256/8G Black بحالة مم...,256.0,8.0,used,L1:slash,text,L2:used
127,سامسونج اس 24 الترا بحالة ممتازة,NaN,NaN,used,NaN,NaN,L2:used
228,Realme gt master Edition 5G zero,NaN,NaN,new,NaN,NaN,L2:new
320,موبايل رينو 14 برو لم يستعمل غير شهر واحد,NaN,NaN,ambiguous,NaN,NaN,L2:new+used
354,Samsung Note 10 Plus_256_12_فررصة الوحيد حالة ...,256.0,NaN,NaN,L2:bare,NaN,NaN
441,سامسونج ام 30 اس. بحالة ممتازة,NaN,NaN,used,NaN,NaN,L2:used
447,"زيرو 256 , 12",256.0,12.0,new,L1:bare,L2:pair,L1
468,Samsung S21 Ultra 256GB 12Ram ب متفتحش قبل كده...,256.0,12.0,new,L1:unit,L2:kw,L2:new
507,S25 Ultra 12+256 لسا علبته متفتحتش معليهوش ضريبة,256.0,12.0,NaN,L1:bare,L2:pair,NaN
545,إكس اس ماكس بشريحتين بحالة ممتازة جدا,NaN,NaN,used,NaN,NaN,L2:used


In [ ]:
display(l2_rows(stage2_val_df)[["title","storage_gb","ram_gb","condition","storage_src","ram_src","condition_src"]].head(15))

,title,storage_gb,ram_gb,condition,storage_src,ram_src,condition_src
44,هواويه ميت 10 لايت بحالة جيدة لم يفتح من قبل,NaN,NaN,new,NaN,NaN,L2:new
74,Realme 9i 6/128 بحالة ممتازة جدًا + الكرتونة,128.0,6.0,used,L1:slash,text,L2:used
447,شاومي ريدمي ١٧ الجيل الرابع ٦ رام ٢٥٦ جيجا متب...,256.0,6.0,new,L1:unit,L2:kw,L1
459,لينوفو كيه 14 بلس وارد الخارج ٦٤جيجا+٤جيجا را...,64.0,6.0,used,L1:unit,text,L2:used
469,إنفنيكس هوت 50 حاله ممتازه جدا,NaN,NaN,used,NaN,NaN,L2:used
559,Ultra256,256.0,NaN,NaN,L2:bare,NaN,NaN
589,هواوي y92019 متفتحش بحالة ممتازة وعلبتة,NaN,NaN,new,NaN,NaN,L2:new
631,honor magic 8 pro 1 tb 16 ram,1024.0,16.0,NaN,L1:unit,L2:kw,NaN
647,Apple - iPhone 16 Plus بطريه 92% بدون اي عيوب ...,NaN,NaN,new,NaN,NaN,L2:new
699,سامسونج ايه 71 بحاله جيده جدا,NaN,NaN,used,NaN,NaN,L2:used


In [ ]:
out_train = STAGE2_DIR / "stage2_train_df.csv"
out_val   = STAGE2_DIR / "stage2_val_df.csv"

stage2_train_df.to_csv(out_train, index=False)
stage2_val_df.to_csv(out_val, index=False)

print("saved", out_train.name, stage2_train_df.shape)
print("saved", out_val.name, stage2_val_df.shape)

saved stage2_train_df.csv (5165, 60)
saved stage2_val_df.csv (1252, 60)
